# 03/06 — Confounder + composition robustness

Sanity layers reviewers will ask about:

1. **Section QC** — drop low-quality sections, re-run T1/T3.
2. **Slide batch** — re-fit signature mixed model with batch as fixed effect; check estimates do not collapse.
3. **Cell composition** — regress out tangram cell-type fractions per spot before computing signature scores; rerun responder map.
4. **Meninges contamination** — compare attenuation in hippocampus with vs without spots within N pixels of the rim.

In [ ]:
from __future__ import annotations
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

ROOT = Path.cwd().resolve()
while not (ROOT / 'utils').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('could not locate project root')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# regional-annotation h5ad lives on the processing volume
BASEDIR = Path('/Volumes/processing2/ST_BRICHOS/data')
H5AD_ORIENTED = BASEDIR / 'ST_BRICHOS_region_subcluster_oriented.h5ad'
H5AD_BASE     = BASEDIR / 'ST_BRICHOS_region_subcluster.h5ad'
H5AD = H5AD_ORIENTED if H5AD_ORIENTED.exists() else H5AD_BASE
COUNT_LAYER = 'counts'   # raw integer counts live here, not in .X

TBL = ROOT / 'results' / 'tables' / 'attenuation'
TBL.mkdir(parents=True, exist_ok=True)
FIG = ROOT / 'results' / 'figures' / 'manuscript'
FIG.mkdir(parents=True, exist_ok=True)

SAMPLE_KEY    = 'sample_id'           # change to 'library_id' if obs uses that
REGION_KEY    = 'anatomical_region'   # adjust to your obs column for regions
TREATMENT_KEY = 'treatment'
print('h5ad        :', H5AD)
print('count layer :', COUNT_LAYER)


### 1. Section QC drop-and-rerun

In [ ]:
adata = sc.read_h5ad(H5AD)
qc = (adata.obs.groupby(SAMPLE_KEY)
      .agg(n_spots=('total_counts','size'),
           median_counts=('total_counts','median'),
           pct_mito=('pct_counts_mt','mean'))
      .sort_values('median_counts'))
qc


In [ ]:
BAD_SAMPLES = []  # fill in based on QC: e.g. ['P28052_103']
if BAD_SAMPLES:
    keep = ~adata.obs[SAMPLE_KEY].isin(BAD_SAMPLES)
    adata_qc = adata[keep].copy()
else:
    adata_qc = adata
print(adata_qc)


### 2. Refit attenuation on QC-filtered cohort

In [ ]:
from utils.attenuation import (
    make_pseudobulk, pseudobulk_lfc, attenuation_slope,
    sign_concordance,
)
counts_qc, meta_qc = make_pseudobulk(adata_qc,
    sample_key=SAMPLE_KEY, region_key=REGION_KEY,
    treatment_key=TREATMENT_KEY, min_spots=20,
    layer=COUNT_LAYER)
counts_qc = counts_qc.round().astype(int)
rows = []
for region in sorted(meta_qc['region'].dropna().unique()):
    try:
        pw = pseudobulk_lfc(counts_qc, meta_qc, 'PBS', 'WT', region)
        bw = pseudobulk_lfc(counts_qc, meta_qc, 'BRICHOS', 'WT', region)
    except ValueError:
        continue
    common = pw.lfc.index.intersection(bw.lfc.index)
    sig = pw.padj.loc[common] < 0.05
    res = attenuation_slope(pw.lfc.loc[common],
                            bw.lfc.loc[common], sig)
    rows.append(dict(region=region, slope_qc=res['slope'],
                     ci_low=res['ci_low'], ci_high=res['ci_high']))
qc_df = pd.DataFrame(rows).set_index('region')
qc_df.to_csv(TBL / 'qc_filtered_attenuation.tsv', sep='\t')
qc_df


### 3. Composition adjustment

Requires a `cell_type_fractions` obsm slot from tangram (or another deconvolution). For each signature score, regress out fractions and rerun the mixed model on residuals.

Skeleton below — fill in once `obsm['cell_type_fractions']` is populated.

In [ ]:
if 'cell_type_fractions' not in adata.obsm:
    print('skip — no obsm[cell_type_fractions] yet')
else:
    from sklearn.linear_model import LinearRegression
    X = adata.obsm['cell_type_fractions']
    for sig in ['PIG_score', 'microglia_score']:
        if sig not in adata.obs.columns:
            continue
        y = adata.obs[sig].values
        keep = np.isfinite(y).all() if y.ndim > 1 else np.isfinite(y)
        lr = LinearRegression().fit(X[keep], y[keep])
        adata.obs[f'{sig}_resid'] = y - lr.predict(X)
        print(f'{sig}: residualised against cell types')


### 4. Meninges-rim sensitivity

Recompute hippocampal attenuation slope with and without spots within `D` pixels of the rim mask. If the slope is sensitive to D, the regional rescue may be partly meningeal contamination.

In [ ]:
# Pseudo-code: assumes obs has 'distance_to_rim_px' from nb 07.
if 'distance_to_rim_px' not in adata.obs.columns:
    print('skip — distance_to_rim_px not yet computed')
else:
    rows = []
    for d in [0, 50, 100, 200, 400]:
        mask = adata.obs['distance_to_rim_px'] >= d
        a = adata[mask].copy()
        c, m = make_pseudobulk(a, sample_key=SAMPLE_KEY,
                                region_key=REGION_KEY,
                                treatment_key=TREATMENT_KEY,
                                min_spots=10,
                                layer=COUNT_LAYER)
        c = c.round().astype(int)
        try:
            pw = pseudobulk_lfc(c, m, 'PBS', 'WT',
                                 region='Hippocampal_formation')
            bw = pseudobulk_lfc(c, m, 'BRICHOS', 'WT',
                                 region='Hippocampal_formation')
        except (KeyError, ValueError):
            continue
        common = pw.lfc.index.intersection(bw.lfc.index)
        sig = pw.padj.loc[common] < 0.05
        res = attenuation_slope(pw.lfc.loc[common],
                                bw.lfc.loc[common], sig)
        rows.append(dict(min_distance_px=d, slope=res['slope'],
                         n=res['n']))
    pd.DataFrame(rows)
